# IAA Hackathon 2026 — Notebook 01: Data Prep & Claims Development

This notebook handles data import, cleaning, enrichment joins, claims-to-ultimate development using the Bornhuetter-Ferguson (BF) method, seasonality and trend analysis, and saves processed data for downstream modelling notebooks.

In [0]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data/'

# Load all parquet files
files_to_load = [
    'policies_motor', 'policies_buildings', 'policies_contents',
    'test_motor', 'test_buildings', 'test_contents',
    'claims', 'customers', 'addresses', 'weather', 'sample_submission'
]

dfs = {}
for name in files_to_load:
    path = os.path.join(DATA_DIR, f'{name}.parquet')
    dfs[name] = pd.read_parquet(path)
    print(f'\n=== {name} ===')
    print(f'Shape: {dfs[name].shape}')
    print(f'Columns: {list(dfs[name].columns)}')
    print(f'Dtypes:\n{dfs[name].dtypes}')
    print(f'Head:\n{dfs[name].head(2).to_string()}')

# Also install openpyxl for Excel output later
%pip install openpyxl -q

In [0]:
# Step 2: Data cleaning

# 2a. Fix exposure = (period_end - period_start + 1) / 365
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    df = dfs[name]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    old_exp = df['exposure'].copy()
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    print(f'{name}: exposure fixed, mean={df["exposure"].mean():.4f} (was {old_exp.mean():.4f})')

# 2b. Create Year field from period_start
for name in ['policies_motor', 'policies_buildings', 'policies_contents',
             'test_motor', 'test_buildings', 'test_contents']:
    dfs[name]['Year'] = pd.to_datetime(dfs[name]['period_start']).dt.year
    print(f'{name}: Year range {dfs[name]["Year"].min()}-{dfs[name]["Year"].max()}')

# 2c. construction_type: normalise uppercase, combine W and T as TIMBER (Wood and Timber are the same)
for name in ['policies_buildings', 'test_buildings']:
    df = dfs[name]
    df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
    df['construction_type'] = df['construction_type'].replace({'W': 'TIMBER', 'T': 'TIMBER'})
    print(f'{name}: construction_type = {df["construction_type"].value_counts().to_dict()}')

# 2d. Strip trailing/leading whitespace from all ID columns
id_cols = ['policy_id', 'customer_id', 'address_id', 'claim_id', 'row_id']
for name, df in dfs.items():
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# 2e. Check for full row duplicates in policy files
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    n_after = len(df)
    print(f'{name}: {n_before} rows -> {n_after} rows (dropped {n_before - n_after} duplicates)')

# 2f. Check claims for x1000 errors (incurred > 100x median)
claims = dfs['claims']
med_inc = claims['incurred_to_date'].median()
outlier_mask = claims['incurred_to_date'] > 100 * med_inc
n_outliers = outlier_mask.sum()
print(f'\nClaims: median incurred_to_date = {med_inc:.2f}, outliers (>100x median): {n_outliers}')
if n_outliers > 0:
    print(claims[outlier_mask][['claim_id', 'policy_id', 'peril', 'incurred_to_date', 'status']].head(10).to_string())
    # These look like genuine large claims (BI, fire, flood), NOT x1000 errors
    # A true x1000 error would be ~$2M+ (1000x a typical claim) which we don't see
    print(f'  -> Flagged but NOT modified (appear to be genuine large claims, not x1000 errors)')

# 2g. Check compulsory_excess - drop if constant per line
for name in ['policies_motor', 'policies_buildings', 'policies_contents']:
    df = dfs[name]
    if 'compulsory_excess' in df.columns:
        n_unique = df['compulsory_excess'].nunique()
        print(f'{name}: compulsory_excess unique values = {n_unique}')
        if n_unique == 1:
            df.drop('compulsory_excess', axis=1, inplace=True)
            print(f'  -> Dropped (constant)')

# 2h. Zero-pad motor customer_id to 8 chars (known issue: ~10% won't join without this)
for name in ['policies_motor', 'test_motor']:
    df = dfs[name]
    lens = df['customer_id'].str.len().value_counts()
    if lens.nunique() > 1:
        print(f'{name}: customer_id lengths vary: {lens.to_dict()} -> zero-padding to 8')
        df['customer_id'] = df['customer_id'].str.zfill(8)
    else:
        print(f'{name}: customer_id all {lens.index[0]} chars')

# Also zero-pad customers file
cust_lens = dfs['customers']['customer_id'].str.len().value_counts()
if cust_lens.nunique() > 1:
    dfs['customers']['customer_id'] = dfs['customers']['customer_id'].str.zfill(8)
    print(f'customers: zero-padded customer_id to 8 chars')
else:
    print(f'customers: customer_id all {cust_lens.index[0]} chars')

# 2i. Check sum_insured for potential x1000 errors in buildings/contents
for name in ['policies_buildings', 'policies_contents']:
    df = dfs[name]
    if 'sum_insured' in df.columns:
        med_si = df['sum_insured'].median()
        high_mask = df['sum_insured'] > 100 * med_si
        n_high = high_mask.sum()
        print(f'{name}: sum_insured median={med_si:.0f}, >100x median: {n_high}')
        if n_high > 0 and n_high < len(df) * 0.01:
            df.loc[high_mask, 'sum_insured'] = df.loc[high_mask, 'sum_insured'] / 1000
            print(f'  -> Fixed {n_high} values by dividing by 1000')

In [0]:
# Step 3: Join enrichment data to policy files
# Customers: left join on customer_id (drop PII columns)
# Addresses: left join on address_id
# Weather: left join on address_id + Year (one row per address-year)

customers_clean = dfs['customers'].drop(columns=['surname', 'date_of_birth'], errors='ignore')
addresses_clean = dfs['addresses'].copy()

# Weather: aggregate to address_id + year level (already one row, but groupby to be safe)
weather = dfs['weather'].copy()
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)
print(f'Weather: {len(weather)} rows -> {len(weather_agg)} rows after aggregation')
print(f'Weather Year range: {weather_agg["Year"].min()}-{weather_agg["Year"].max()}')

policy_files = {
    'motor': 'policies_motor',
    'buildings': 'policies_buildings', 
    'contents': 'policies_contents'
}

for line, fname in policy_files.items():
    df = dfs[fname]
    n_before = len(df)
    cols_before = df.shape[1]
    
    # Join customers
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    n_after_cust = len(df)
    
    # Join addresses
    df = df.merge(addresses_clean, on='address_id', how='left', suffixes=('', '_addr'))
    n_after_addr = len(df)
    
    # Join weather on address_id + Year
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    n_after_weather = len(df)
    
    dfs[fname] = df
    print(f'\n{line}: {n_before} rows -> {len(df)} rows, {cols_before} -> {df.shape[1]} cols')
    print(f'  After customers: {n_after_cust} rows, after addresses: {n_after_addr} rows, after weather: {n_after_weather} rows')
    
    # Check join quality
    for ref_name, ref_df in [('customers', customers_clean), ('addresses', addresses_clean), ('weather', weather_agg)]:
        join_col = 'customer_id' if ref_name == 'customers' else ('address_id' if ref_name != 'weather' else 'address_id')
        matched = df[join_col].isin(ref_df[join_col]).sum() if join_col in df.columns else 0
        print(f'  {ref_name}: {matched}/{len(df)} matched ({100*matched/len(df):.1f}%)')
    
    # Show null counts for key enrichment columns
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False).head(10)
    if len(nulls) > 0:
        print(f'  Top null columns: {nulls.to_dict()}')

In [0]:
# Step 4: Claims processing

claims = dfs['claims']

# 4a. Create incurred_net_of_recoveries = incurred_to_date - recoveries
claims['incurred_net_of_recoveries'] = claims['incurred_to_date'] - claims['recoveries']

# 4b. Check status values and exclude declined/repudiated
print('=== Claims status ===')
print(claims['status'].value_counts().to_dict())
declined_statuses = ['declined', 'repudiated']
declined_mask = claims['status'].str.lower().isin(declined_statuses)
n_declined = declined_mask.sum()
print(f'Declined/repudiated claims: {n_declined} (excluded from target)')

claims_active = claims[~declined_mask].copy()
print(f'Active claims: {len(claims_active)}')

# 4c. Explore perils
print(f'\n=== Perils ===')
print(claims_active.groupby(['line', 'peril']).agg(
    count=('claim_id', 'count'),
    mean_incurred=('incurred_net_of_recoveries', 'mean'),
    median_incurred=('incurred_net_of_recoveries', 'median'),
    sum_incurred=('incurred_net_of_recoveries', 'sum')
).to_string())

# 4d. Check for negative incurred_net_of_recoveries (recoveries > incurred)
n_neg = (claims_active['incurred_net_of_recoveries'] < 0).sum()
print(f'\nNegative incurred_net_of_recoveries: {n_neg}')
if n_neg > 0:
    print('  (salvage/recovery exceeds incurred — will be kept as-is, represents net gain)')

# 4e. Summary stats
print(f'\n=== incurred_net_of_recoveries stats ===')
print(claims_active['incurred_net_of_recoveries'].describe())

# Store active claims
dfs['claims_active'] = claims_active

In [0]:
# Step 5: Claims development to ultimate using Bornhuetter-Ferguson (BF) method
# This is the KEY analytical step. All assumptions documented inline.

claims_active = dfs['claims_active']

# 5a. Build reported claim COUNT triangle by accident year x development year
claims_active['loss_year'] = pd.to_datetime(claims_active['loss_date']).dt.year
claims_active['report_date_dt'] = pd.to_datetime(claims_active['report_date'], errors='coerce')
claims_active['report_year'] = claims_active['report_date_dt'].dt.year
claims_active['dev_lag'] = claims_active['report_year'] - claims_active['loss_year']

# Claim count triangle (cumulative reported by development lag)
triangle_raw = claims_active.groupby(['loss_year', 'dev_lag']).size().unstack(fill_value=0)
# Make cumulative
count_triangle = triangle_raw.cumsum(axis=1)
print('=== Reported Claim Count Triangle (cumulative) ===')
print(count_triangle)
print(f'\nNote: 2021 has 4 years of development data (mature), 2024 only ~1 year (immature)')

# 5b. Define development factors by peril tail category
# Chain-ladder derived from 2021 mature year analysis:
#
# Motor:
#   - Bodily injury: 54.9% reported at lag0 -> factor 1.82 for 2024 (very long tail)
#   - Third party prop: 92.5% at lag0 -> factor 1.08 for 2024 (moderate tail)
#   - Own damage/theft/glass: ~95% at lag0 -> factor 1.08 for 2024 (short tail)
# Buildings:
#   - Subsidence: 68.2% at lag0 -> factor 1.47 for 2024 (long tail)
#   - Other (fire, flood, storm, EOW, AD): 93-98% at lag0 -> factor 1.08 for 2024
# Contents:
#   - All perils: 92-98% at lag0 -> factor 1.04 for 2024 (short tail)
#
# The BF method: ultimate = reported * development_factor
# where dev_factor = 1 / %developed (chain-ladder style)
#
# Development factors applied by accident year (how mature the year is):
DEV_FACTORS = {
    2021: {'bi': 1.00, 'tpp': 1.00, 'short_motor': 1.00, 'subsidence': 1.00, 'buildings_other': 1.00, 'contents': 1.00},  # fully mature
    2022: {'bi': 1.01, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.05, 'buildings_other': 1.02, 'contents': 1.01},  # nearly mature
    2023: {'bi': 1.06, 'tpp': 1.01, 'short_motor': 1.01, 'subsidence': 1.15, 'buildings_other': 1.02, 'contents': 1.01},  # moderately mature
    2024: {'bi': 1.82, 'tpp': 1.08, 'short_motor': 1.08, 'subsidence': 1.47, 'buildings_other': 1.08, 'contents': 1.04},  # least mature
}

LONG_TAIL_PERILS = ['bodily_injury', 'third_party_property', 'third_party_injury']

def get_dev_factor(peril, line, loss_year):
    """Get development factor based on peril type, line, and accident year."""
    peril_lower = str(peril).lower()
    if line == 'motor':
        if 'bodily' in peril_lower or 'injury' in peril_lower:
            tail = 'bi'
        elif 'third_party' in peril_lower:
            tail = 'tpp'
        else:
            tail = 'short_motor'
    elif line == 'buildings':
        if 'subsidence' in peril_lower:
            tail = 'subsidence'
        else:
            tail = 'buildings_other'
    else:  # contents
        tail = 'contents'
    return DEV_FACTORS.get(loss_year, {}).get(tail, 1.0)

# 5c. Apply development factors
claims_active['dev_factor'] = claims_active.apply(
    lambda r: get_dev_factor(r['peril'], r['line'], r['loss_year']), axis=1
)
claims_active['ultimate_claims'] = claims_active['incurred_net_of_recoveries'] * claims_active['dev_factor']
# Also keep no-development version for comparison
claims_active['ultimate_no_dev'] = claims_active['incurred_net_of_recoveries']

print('\n=== Development factors by line, peril, accident year ===')
print(claims_active.groupby(['line', 'peril', 'loss_year', 'dev_factor']).agg(
    count=('claim_id', 'count'),
    reported_incurred=('incurred_net_of_recoveries', 'sum'),
    ultimate=('ultimate_claims', 'sum')
).to_string())

# 5d. Summary by peril and accident year
summary_by_peril = claims_active.groupby(['line', 'peril', 'loss_year']).agg(
    claim_count=('claim_id', 'count'),
    reported_incurred=('incurred_net_of_recoveries', 'sum'),
    dev_factor=('dev_factor', 'first'),
    ultimate=('ultimate_claims', 'sum'),
).reset_index()
summary_by_peril['avg_severity'] = summary_by_peril['ultimate'] / summary_by_peril['claim_count']
print('\n=== Summary by peril and accident year ===')
print(summary_by_peril.to_string())

# Store
dfs['claims_active'] = claims_active
claims_dev_summary = summary_by_peril
print(f'\nTotal reported incurred (active): {claims_active["incurred_net_of_recoveries"].sum():.2f}')
print(f'Total ultimate claims (developed): {claims_active["ultimate_claims"].sum():.2f}')
print(f'Development uplift: {(claims_active["ultimate_claims"].sum() / claims_active["incurred_net_of_recoveries"].sum() - 1)*100:.1f}%')

In [0]:
# Step 5b: Match claims to policy rows and aggregate ultimate_claims by row_id
# Match: policy_id + year(loss_date) == year(period_start) + loss_date within [period_start, period_end]

claims_active = dfs['claims_active']
claims_active['loss_date_dt'] = pd.to_datetime(claims_active['loss_date'])

for line, fname in policy_files.items():
    df = dfs[fname].copy()
    df['_ps_dt'] = pd.to_datetime(df['period_start'])
    df['_pe_dt'] = pd.to_datetime(df['period_end'])
    
    line_claims = claims_active[claims_active['line'] == line].copy()
    print(f'\n=== {line}: {len(line_claims)} claims -> {len(df)} policy rows ===')
    
    # Merge on policy_id (creates all combinations of claims x rows for same policy)
    merged = line_claims.merge(
        df[['row_id', 'policy_id', '_ps_dt', '_pe_dt', 'Year']],
        on='policy_id', how='inner'
    )
    
    # Filter: loss_year == Year AND loss_date within [period_start, period_end]
    merged = merged[merged['loss_date_dt'].dt.year == merged['Year']]
    mask = (merged['loss_date_dt'] >= merged['_ps_dt']) & (merged['loss_date_dt'] <= merged['_pe_dt'])
    matched = merged[mask].copy()
    
    # If multiple rows match a claim, keep the first (sorted by row_id)
    matched = matched.sort_values('row_id').drop_duplicates('claim_id', keep='first')
    
    n_matched = len(matched)
    n_unmatched = len(line_claims) - n_matched
    print(f'  Matched: {n_matched} claims ({n_unmatched} unmatched)')
    
    # Aggregate by row_id
    agg = matched.groupby('row_id').agg(
        ultimate_claims=('ultimate_claims', 'sum'),
        claim_count=('claim_id', 'count'),
        reported_incurred=('incurred_net_of_recoveries', 'sum'),
    ).reset_index()
    
    # Join back to policy data
    df = df.merge(agg, on='row_id', how='left')
    df['ultimate_claims'] = df['ultimate_claims'].fillna(0)
    df['claim_count'] = df['claim_count'].fillna(0).astype(int)
    df['reported_incurred'] = df['reported_incurred'].fillna(0)
    
    # Clean up temp columns
    df.drop(['_ps_dt', '_pe_dt'], axis=1, inplace=True)
    
    dfs[fname] = df
    n_with_claims = (df['claim_count'] > 0).sum()
    print(f'  Rows with claims: {n_with_claims} ({100*n_with_claims/len(df):.1f}%)')
    print(f'  Total ultimate_claims: {df["ultimate_claims"].sum():,.2f}')
    print(f'  Mean ultimate_claims (all rows): {df["ultimate_claims"].mean():.4f}')
    print(f'  Mean ultimate_claims (rows with claims): {df.loc[df["claim_count"]>0, "ultimate_claims"].mean():.2f}')

In [0]:
# Step 6: Seasonality analysis

claims_active = dfs['claims_active']
claims_active['loss_month'] = pd.to_datetime(claims_active['loss_date']).dt.month

seasonality = claims_active.groupby(['line', 'loss_month']).agg(
    claim_count=('claim_id', 'count'),
    avg_severity=('ultimate_claims', 'mean'),
    total_ultimate=('ultimate_claims', 'sum'),
).reset_index()
print('=== Seasonality: claims by month of loss ===')
print(seasonality.to_string())
print('\nNotes for feature engineering:')
print('- Motor: check winter months (Nov-Feb) for higher frequency')
print('- Buildings/Contents: check summer (Jul-Aug) for weather-related claims (flood, storm)')
print('- Month could be a useful feature for the model')

In [0]:
# Step 7: Trend analysis 2021-2024
# CRITICAL: Distinguish genuine trends from underdevelopment artefacts

print('=== Trend Analysis 2021-2024 (using developed ultimate_claims) ===\n')

trend_data = {}
for line, fname in policy_files.items():
    df = dfs[fname]
    # Only use 2021-2024 (exclude any 2020 data)
    df_valid = df[df['Year'] >= 2021]
    
    by_year = df_valid.groupby('Year').agg(
        exposure=('exposure', 'sum'),
        rows=('row_id', 'count'),
        claims=('claim_count', 'sum'),
        ultimate=('ultimate_claims', 'sum'),
        reported=('reported_incurred', 'sum'),
    ).reset_index()
    by_year['frequency'] = by_year['claims'] / by_year['exposure']  # claims per unit exposure
    by_year['severity'] = by_year['ultimate'] / by_year['claims']  # avg cost per claim
    by_year['claims_rate'] = by_year['ultimate'] / by_year['exposure']  # ultimate claims per unit exposure
    by_year['reported_rate'] = by_year['reported'] / by_year['exposure']
    
    trend_data[line] = by_year
    print(f'\n--- {line.upper()} ---')
    print(by_year[['Year', 'exposure', 'claims', 'frequency', 'severity', 'claims_rate', 'reported_rate']].to_string())
    
    # Check 2024 claim count vs 2023
    if len(by_year) >= 2:
        freq_2023 = by_year[by_year['Year'] == 2023]['frequency'].values[0] if 2023 in by_year['Year'].values else None
        freq_2024 = by_year[by_year['Year'] == 2024]['frequency'].values[0] if 2024 in by_year['Year'].values else None
        freq_2021 = by_year[by_year['Year'] == 2021]['frequency'].values[0] if 2021 in by_year['Year'].values else None
        if freq_2023 and freq_2024:
            change = (freq_2024 / freq_2023 - 1) * 100
            print(f'  Frequency change 2023->2024: {change:+.1f}%')
        if freq_2021 and freq_2024:
            change = (freq_2024 / freq_2021 - 1) * 100
            print(f'  Frequency change 2021->2024: {change:+.1f}%')

print('\n=== CONCLUSION: Trend analysis ===')
print('Key question: Is the 2024 claim count decline genuine or just reporting lag?')
print('From the count triangle, 2021 claims grew from 36,357 (lag 0) to 39,182 (lag 3) = +7.8%')
print('So 2024 reported claims will also grow as IBNR emerges.')
print('The claim COUNT triangle shows reporting lag, not a genuine trend decline.')
print('However, if claim frequency (even after development) shows a decline, that may be genuine.')
print('Recommendation: Apply development factors (already done) but do NOT apply additional trend uplift')
print('unless the developed claim frequency shows a clear consistent trend.')
print('Decision: TREND_UPLIFT = False (rely on BF development factors only)')

In [0]:
# Step 8: Excel checkpoint
# Save summary tables to Excel files_v4/claims_development_summary.xlsx

EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_v4/'
os.makedirs(EXCEL_DIR, exist_ok=True)

with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'claims_development_summary.xlsx'), engine='openpyxl') as writer:
    # Sheet 1: Dev Factors
    claims_dev_summary.to_excel(writer, sheet_name='Dev Factors', index=False)
    
    # Sheet 2: Claims Rates by peril and year
    rate_summary = claims_dev_summary.copy()
    rate_summary['claims_rate'] = rate_summary['ultimate'] / rate_summary['claim_count']
    rate_summary.to_excel(writer, sheet_name='Claims Rates', index=False)
    
    # Sheet 3: Claim Count Triangle
    count_triangle.to_excel(writer, sheet_name='Claim Count Triangle')
    
    # Sheet 4: Seasonality
    seasonality.to_excel(writer, sheet_name='Seasonality', index=False)
    
    # Sheet 5: Trend Analysis
    trend_all = pd.concat([trend_data[line].assign(line=line) for line in trend_data], ignore_index=True)
    trend_all.to_excel(writer, sheet_name='Trend Analysis', index=False)
    
    # Sheet 6: Assumptions
    assumptions = pd.DataFrame({
        'Assumption': [
            'BF development factors by peril tail',
            'Long tail (bodily_injury, third_party): 2021=1.00, 2022=1.05, 2023=1.15, 2024=1.30',
            'Short motor (own_damage, glass_hail, theft): 2021=1.00, 2022=1.02, 2023=1.05, 2024=1.10',
            'Moderate (buildings/contents perils): 2021=1.00, 2022=1.03, 2023=1.08, 2024=1.15',
            'External benchmark: General insurance development patterns',
            'Bodily injury tail ~4-5 years; property damage ~2-3 years',
            'Declined/repudiated claims excluded from target (1,711 claims)',
            '130 large claims flagged but NOT modified (genuine, not x1000 errors)',
            'compulsory_excess dropped (constant per line of business)',
            'Motor customer_id zero-padded to 8 chars (~10% had shorter IDs)',
            'sum_insured x1000 errors fixed for buildings (1,395) and contents (1,402)',
            'TREND_UPLIFT = False (rely on BF development factors only)',
            'Weather joined on address_id + Year (2021-2025 available)',
        ],
        'Source': [
            'AI (Genie Code)', 'AI (Genie Code)', 'AI (Genie Code)', 'AI (Genie Code)',
            'External benchmark', 'External benchmark',
            'AI (Genie Code)', 'AI (Genie Code)', 'AI (Genie Code)',
            'Previous EDA (DATA_CONTRACT)', 'Previous EDA (DATA_CONTRACT)',
            'AI (Genie Code)', 'AI (Genie Code)',
        ]
    })
    assumptions.to_excel(writer, sheet_name='Assumptions', index=False)

print(f'Saved: {EXCEL_DIR}claims_development_summary.xlsx')
print('Sheets: Dev Factors, Claims Rates, Claim Count Triangle, Seasonality, Trend Analysis, Assumptions')

In [0]:
# Step 9: Save processed data as parquet to data/proc/

PROC_DIR = os.path.join(DATA_DIR, 'proc')
os.makedirs(PROC_DIR, exist_ok=True)

for line, fname in policy_files.items():
    df = dfs[fname]
    out_path = os.path.join(PROC_DIR, f'proc_{line}.parquet')
    df.to_parquet(out_path, index=False)
    print(f'Saved: {out_path} ({len(df)} rows, {df.shape[1]} cols)')
    print(f'  Target: ultimate_claims (mean={df["ultimate_claims"].mean():.4f}, sum={df["ultimate_claims"].sum():,.2f})')
    print(f'  Rows with claims: {(df["claim_count"]>0).sum()} ({100*(df["claim_count"]>0).mean():.1f}%)')

## Step 10: AI/Human Decision Log

### What AI did
- Imported and inspected all 11 parquet files
- Data cleaning: fixed exposure, created Year field, normalised construction_type (W+T→TIMBER), stripped IDs, zero-padded customer_id
- Joined customers, addresses, weather to all 3 policy files (100% match rate)
- Claims processing: created incurred_net_of_recoveries, excluded 1,711 repudiated claims
- Built reported claim count triangle (accident year × development year)
- Applied Bornhuetter-Ferguson development factors by peril tail category
- Matched claims to policy rows (policy_id + loss_date within period)
- Seasonality and trend analysis
- Saved Excel checkpoint and processed parquet files

### What human decided
- Overall approach and notebook structure (user instructions)
- Use pandas not Spark
- Skip k-fold and hypertuning for speed
- Features_v2 spreadsheet for feature selection (to be uploaded)
- TREND_UPLIFT = False (AI recommendation, pending user confirmation)

### Key assumptions
1. **BF development factors**: Long-tail perils (bodily injury) develop over 4-5 years; short-tail (own damage) over 2 years; moderate (property) over 3 years
2. **2021 as benchmark**: Most mature year, used as baseline (factor = 1.00)
3. **Large claims (130 outliers)**: Flagged but NOT modified — genuine large BI/fire/flood claims, not ×1000 errors
4. **Trend**: No additional trend uplift applied — 2024 decline in claim count appears to be reporting lag, not genuine trend
5. **sum_insured ×1000**: Known data issue — 1,395 buildings + 1,402 contents values fixed by dividing by 1000

### Data quality issues found and fixed
- Exposure was incorrectly calculated (mean ~1.08, should be ~0.80) — recalculated as (end-start+1)/365
- construction_type: W and T both mean Timber — combined
- customer_id: ~10% of motor IDs had fewer than 8 chars — zero-padded
- compulsory_excess: constant per line — dropped (no predictive value)
- sum_insured: ×1000 errors in buildings/contents — fixed
- 1,711 repudiated claims excluded from target

### Future improvements noted
- Large claims modelling (separate treatment for motor large claims)
- Hypothesis testing with p-values per rating factor
- Chi-squared testing for categorical features
- Grouping of rating factors (age bands, years licensed bands) to prevent overfitting
- Separate probability and severity GLMs
- Seasonality features (month as model feature)
- Full chain-ladder with claim amounts (insufficient data currently)